## Q-Mini 0.7M — Existing Drive Data + Offline Chat (CPU only)

[Open directly in Google Colab](https://colab.research.google.com/github/lion36508/Q-mini-light-0.7M/blob/main/colab/QMini_Existing_Drive_Data_CPU_Chat.ipynb)

यह notebook पहले से Google Drive में मौजूद data का उपयोग करेगा। Mistral, API keys, paid API या नए paid datasets की जरूरत नहीं है।

- CPU-only; GPU इस्तेमाल नहीं होगा।
- पहले से मौजूद human English conversation data (लगभग 9,429 examples) reuse होगा; matching पुराने SFT weights मिलने पर warm-start होगा।
- Drive की किताबें/PDF/DOCX/TXT/Markdown/CSV/JSONL/Parquet files को local searchable library में index करेगा।
- Chat के समय किताबों से relevant अंश खोजकर Q-Mini को देगा।
- पुरानी files delete/overwrite नहीं होंगी; नया output अलग folder में बनेगा।
- English-first training; दूसरी भाषाओं की किताबें searchable रहेंगी, पर English training data में मिलाई नहीं जाएँगी।

ऊपर Runtime → Run all दबाएँ, Drive permission दें, और अंत में Q-Mini से चैट करें।

In [ ]:
# CPU-only setup
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
import sys, subprocess, importlib.util, json, hashlib, random, time, shutil, math, re

missing = [name for name in ("tokenizers",) if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

import torch
if torch.cuda.is_available():
    raise RuntimeError("CPU-only guard failed. Select Runtime → Change runtime type → CPU and restart.")
torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

REPO_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
REPO_DIR = Path("/content/Q-mini-light-0.7M")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)

sys.path.insert(0, str(REPO_DIR))
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from qmini.config import QMiniConfig
from qmini.tokenizer import load_tokenizer
from qmini.model import QMiniMoE
from qmini.text import escape_special_token_literals
from qmini.export import export_packed_state, verify_packed_export
from qmini.generate import generate, load_model

PROJECT_DRIVE = Path("/content/drive/MyDrive/QMini-Light-0.7M")
PACKAGE_NAME = "QMini-English-512-Q2v2-e1a187f0ca1567cd"
PACKAGE_DIR = PROJECT_DRIVE / "training_ready" / PACKAGE_NAME
BASE_CHECKPOINT = PACKAGE_DIR / "training_state" / "checkpoint-step-000000006530.pt"
TOKENIZER_PATH = PACKAGE_DIR / "tokenizer" / "tokenizer.json"

config_candidates = [
    PACKAGE_DIR / "active_config.json",
    PACKAGE_DIR / "initial_brain" / "config.json",
    REPO_DIR / "configs" / "qmini_light_0.7m.json",
]
CONFIG_PATH = next((p for p in config_candidates if p.is_file()), None)
if not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Expected trained checkpoint not found: {BASE_CHECKPOINT}")
if not TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f"Matching tokenizer not found: {TOKENIZER_PATH}")
if CONFIG_PATH is None:
    raise FileNotFoundError("Matching Q-Mini config.json was not found.")

CONFIG = QMiniConfig.from_json(CONFIG_PATH)
TOKENIZER = load_tokenizer(TOKENIZER_PATH)
if CONFIG.estimated_parameter_count() != 705742 or CONFIG.max_context != 512:
    raise ValueError("Selected package is not the expected Q-Mini Light 0.7M / context-512 profile.")
if TOKENIZER.get_vocab_size() != CONFIG.vocab_size:
    raise ValueError("Tokenizer vocabulary does not match the selected model config.")

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

base_checkpoint = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=True)
if base_checkpoint.get("format") != "qmini-checkpoint-v2":
    raise ValueError("Selected base checkpoint is not a verified Q-Mini v2 checkpoint.")
if QMiniConfig.from_dict(base_checkpoint["config"]).to_dict() != CONFIG.to_dict():
    raise ValueError("Base checkpoint config does not match the package config.")
BASE_MODEL_SHA = sha256_file(BASE_CHECKPOINT)
TOKENIZER_SHA = sha256_file(TOKENIZER_PATH)

RUN_NAME = "existing_drive_data_english_cpu_v1"
RUN_DIR = PROJECT_DRIVE / "assistant_sft_cpu_runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_PATH = RUN_DIR / "mistral_teacher_distillation.jsonl"
HUMAN_PATH = (
    PROJECT_DRIVE / "assistant_sft_cpu_runs"
    / "assistant_sft_oasst1_human_english_v1"
    / "qmini_oasst1_human_english.jsonl"
)
OLD_SFT_DIR = (
    PROJECT_DRIVE / "assistant_sft_cpu_runs"
    / "assistant_sft_oasst1_human_english_v1"
)

SYSTEM_PROMPT = (
    "You are Q-mini, a friendly, practical and accurate personal AI assistant. "
    "Reply naturally and directly. Answer the user's actual question, ask a short "
    "clarifying question when needed, and do not tell stories unless explicitly asked."
)

print("Selected base checkpoint:", BASE_CHECKPOINT)
print("Base checkpoint SHA-256:", BASE_MODEL_SHA)
print("Parameters:", f"{CONFIG.estimated_parameter_count():,}")
print("Context:", CONFIG.max_context)
print("Device: CPU only")


In [ ]:
# Load the existing human-written OASST1 conversations.
# If the saved v1 dataset exists, reuse it instead of downloading again.

if HUMAN_PATH.is_file():
    human_records = [
        json.loads(line)
        for line in HUMAN_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print("Reused saved human OASST1 data:", HUMAN_PATH)
else:
    print("Saved human JSONL not found; downloading OpenAssistant/oasst1.")
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "datasets"], check=True)
        from datasets import load_dataset

    ds = load_dataset("OpenAssistant/oasst1")
    by_id = {}
    for split_name in ds.keys():
        for row in ds[split_name]:
            if row.get("lang") != "en":
                continue
            if row.get("tree_state") != "ready_for_export":
                continue
            if row.get("review_result") is not True:
                continue
            if row.get("deleted") is not False or row.get("synthetic") is not False:
                continue
            if row.get("role") not in ("prompter", "assistant"):
                continue
            text_value = re.sub(r"\s+", " ", str(row.get("text") or "")).strip()
            if text_value:
                clean = dict(row)
                clean["text"] = text_value
                by_id[clean["message_id"]] = clean

    human_records = []
    seen = set()
    for assistant in by_id.values():
        if assistant["role"] != "assistant":
            continue
        rank = assistant.get("rank")
        if rank is not None and int(rank) != 0:
            continue
        chain, visited = [], set()
        current, complete = assistant, True
        while current is not None:
            mid = current.get("message_id")
            if mid in visited:
                complete = False
                break
            visited.add(mid)
            chain.append(current)
            parent = current.get("parent_id")
            current = by_id.get(parent) if parent else None
            if parent and current is None:
                complete = False
                break
        if not complete:
            continue
        chain.reverse()
        tree_id = assistant.get("message_tree_id")
        if not tree_id or not chain or chain[0].get("role") != "prompter":
            continue
        if chain[-1].get("role") != "assistant" or len(chain) % 2:
            continue
        expected = ["prompter" if i % 2 == 0 else "assistant" for i in range(len(chain))]
        if [m["role"] for m in chain] != expected or any(m.get("message_tree_id") != tree_id for m in chain):
            continue
        if assistant["message_id"] in seen:
            continue
        chat = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in chain[-6:]:
            chat.append({
                "role": "user" if msg["role"] == "prompter" else "assistant",
                "content": msg["text"],
            })
        human_records.append({
            "messages": chat,
            "metadata": {
                "source": "OpenAssistant/oasst1",
                "synthetic": False,
                "seed_group": str(tree_id),
                "assistant_message_id": assistant["message_id"],
            },
        })
        seen.add(assistant["message_id"])

    if len(human_records) < 1000:
        raise RuntimeError(f"Only {len(human_records)} valid human records were built; refusing a tiny dataset.")
    HUMAN_PATH.parent.mkdir(parents=True, exist_ok=True)
    with HUMAN_PATH.open("w", encoding="utf-8") as f:
        for row in human_records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

if len(human_records) < 1000:
    raise RuntimeError(f"Human data count is too small: {len(human_records)}")
if not all(row.get("metadata", {}).get("synthetic") is False for row in human_records):
    raise ValueError("Human dataset contains rows not explicitly marked non-synthetic.")

print("Human conversation examples:", len(human_records))
print("Distinct human conversation trees:", len({r["metadata"]["seed_group"] for r in human_records}))
print("Sample last user message:", next(m["content"][:220] for m in reversed(human_records[0]["messages"]) if m["role"] == "user"))


In [ ]:
# External teacher/API disabled. Use only saved human conversations.
teacher_records = []
print('Human examples:', len(human_records))
print('Teacher/API examples: 0')
print('External API calls: disabled')

In [ ]:
# Prepare mixed human + Mistral distillation samples and train on CPU.
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

TARGET_STEPS = 2000
PHYSICAL_BATCH = 4
GRAD_ACCUM = 2
EFFECTIVE_BATCH = PHYSICAL_BATCH * GRAD_ACCUM
CONTEXT_LENGTH = 512
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.05
VALID_FRACTION = 0.10
SEED = 42

all_records = human_records + teacher_records
if len(all_records) < 1000:
    raise RuntimeError("Not enough training records; refusing to train on a tiny dataset.")

# Group all human/teacher variants by original conversation tree to prevent leakage.
groups = {}
for index, row in enumerate(all_records):
    group = row.get("metadata", {}).get("seed_group", f"row:{index}")
    groups.setdefault(str(group), []).append(index)

group_names = list(groups)
random.Random(SEED).shuffle(group_names)
n_valid = max(1, round(len(group_names) * VALID_FRACTION))
valid_groups = set(group_names[:n_valid])
valid_ids = {i for name in valid_groups for i in groups[name]}
train_records = [row for i, row in enumerate(all_records) if i not in valid_ids]
valid_records = [row for i, row in enumerate(all_records) if i in valid_ids]

special_names = ("<pad>", "<bos>", "<eos>", "<|system|>", "<|user|>", "<|assistant|>", "<|end|>")
special = {name: TOKENIZER.token_to_id(name) for name in special_names}
if any(value is None for value in special.values()):
    raise ValueError("Matching Q-Mini tokenizer is missing required chat tokens.")
special = {key: int(value) for key, value in special.items()}

class AssistantChatDataset(Dataset):
    def __init__(self, rows):
        self.samples = []
        role_tokens = {
            "system": special["<|system|>"],
            "user": special["<|user|>"],
            "assistant": special["<|assistant|>"],
        }
        for row in rows:
            messages = row.get("messages", [])
            system_message = next((m for m in messages if m.get("role") == "system"), None)
            if system_message is None:
                continue
            system_ids = TOKENIZER.encode(
                escape_special_token_literals(str(system_message.get("content", SYSTEM_PROMPT)))
            ).ids[:64]
            conversation = [m for m in messages if m.get("role") in ("user", "assistant")][-6:]
            if not conversation or conversation[-1].get("role") != "assistant":
                continue
            if conversation[0].get("role") == "assistant":
                conversation = conversation[1:]
            if not conversation or conversation[0].get("role") != "user":
                continue

            encoded = []
            for message in conversation:
                role = message["role"]
                token_ids = TOKENIZER.encode(
                    escape_special_token_literals(str(message.get("content", "")))
                ).ids
                if len(token_ids) > 192:
                    token_ids = token_ids[:192] if role == "assistant" else token_ids[-192:]
                encoded.append((role, token_ids))

            def sample_length(items):
                return 1 + len(system_ids) + 2 + sum(len(tokens) + 2 for _, tokens in items)

            while len(encoded) > 2 and sample_length(encoded) > CONTEXT_LENGTH:
                encoded = encoded[2:]
                if encoded and encoded[0][0] == "assistant":
                    encoded = encoded[1:]
            if not encoded or encoded[-1][0] != "assistant" or sample_length(encoded) > CONTEXT_LENGTH:
                continue

            ids = [special["<bos>"]]
            mask = [0]
            ids += [role_tokens["system"]] + system_ids + [special["<|end|>"]]
            mask += [0] * (len(system_ids) + 2)

            for role, token_ids in encoded:
                ids += [role_tokens[role]] + token_ids + [special["<|end|>"]]
                if role == "assistant":
                    # Correct target alignment: supervise role marker, response tokens, and turn-ending token.
                    mask += [1] * (len(token_ids) + 2)
                else:
                    mask += [0] * (len(token_ids) + 2)

            if len(ids) > CONTEXT_LENGTH or len(ids) != len(mask):
                raise ValueError("Invalid sample length or loss-mask alignment.")
            if ids[-1] != special["<|end|>"] or mask[-1] != 1:
                raise RuntimeError("Assistant <|end|> supervision invariant failed.")
            self.samples.append((ids, mask))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

def collate_chat(batch):
    max_len = max(len(ids) for ids, _ in batch)
    x = torch.full((len(batch), max_len - 1), special["<pad>"], dtype=torch.long)
    y = torch.full((len(batch), max_len - 1), -100, dtype=torch.long)
    for i, (id_list, mask_list) in enumerate(batch):
        ids = torch.tensor(id_list, dtype=torch.long)
        n = len(ids) - 1
        target_mask = torch.tensor(mask_list[1:], dtype=torch.bool)
        x[i, :n] = ids[:-1]
        labels = ids[1:]
        y[i, :n] = torch.where(target_mask, labels, torch.full_like(labels, -100))
    return {"input_ids": x, "labels": y}

train_data = AssistantChatDataset(train_records)
valid_data = AssistantChatDataset(valid_records)
if len(train_data) < 8 or not len(valid_data):
    raise RuntimeError("Train/validation split is too small.")
print("End-token mask: PASS")
print("Train samples:", len(train_data), "| validation samples:", len(valid_data))
print("Human samples:", len(human_records), "| paid/API teacher samples: 0")

# Warm-start from the prior SFT weights only if base checkpoint, tokenizer, and config all match.
# The old optimizer is intentionally not reused because the training target/loss mask changed.
initial_state = base_checkpoint["model"]
warm_start_path = OLD_SFT_DIR / "assistant_sft_model.pt"
warm_manifest_path = OLD_SFT_DIR / "assistant_sft_manifest.json"
warm_config_path = OLD_SFT_DIR / "config.json"
warm_tokenizer_path = OLD_SFT_DIR / "tokenizer.json"
warm_start_used = False

if all(p.is_file() for p in (warm_start_path, warm_manifest_path, warm_config_path, warm_tokenizer_path)):
    warm_manifest = json.loads(warm_manifest_path.read_text(encoding="utf-8"))
    warm_config = QMiniConfig.from_json(warm_config_path)
    warm_ok = (
        warm_config.to_dict() == CONFIG.to_dict()
        and warm_manifest.get("base_model_sha256") == BASE_MODEL_SHA
        and warm_manifest.get("tokenizer_sha256") == TOKENIZER_SHA
        and sha256_file(warm_tokenizer_path) == TOKENIZER_SHA
    )
    if warm_ok:
        initial_state = torch.load(warm_start_path, map_location="cpu", weights_only=True)
        warm_start_used = True
        print("Warm-started from verified previous SFT model weights.")
    else:
        print("Previous SFT weights did not match base/tokenizer/config; using step-6530 base checkpoint.")

model = QMiniMoE(CONFIG).to("cpu")
model.load_state_dict(initial_state, strict=True)
model.train()

DATASET_SHA = hashlib.sha256(
    (sha256_file(HUMAN_PATH) + (sha256_file(TEACHER_PATH) if TEACHER_PATH.is_file() else "no_teacher_data")).encode()
).hexdigest()
TOKENIZER_SHA = TOKENIZER_SHA
INITIAL_WEIGHTS_SHA = sha256_file(warm_start_path) if warm_start_used else BASE_MODEL_SHA

checkpoint_path = RUN_DIR / "assistant_distill_latest.pt"
metrics_path = RUN_DIR / "assistant_distill_metrics.jsonl"
start_step = 0
best_validation = float("inf")
best_step = 0
stale_evaluations = 0
best_model_state = None
EARLY_STOP_PATIENCE = 8  # stop after 8 evaluations without a new best
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
loader_gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_data, batch_size=PHYSICAL_BATCH, shuffle=True, drop_last=True,
    num_workers=0, collate_fn=collate_chat, generator=loader_gen
)
valid_loader = DataLoader(
    valid_data, batch_size=PHYSICAL_BATCH, shuffle=False, drop_last=False,
    num_workers=0, collate_fn=collate_chat
)

if checkpoint_path.is_file():
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    contract = {
        "format": "qmini-existing-drive-sft-v1",
        "data_sha": DATASET_SHA,
        "tokenizer_sha": TOKENIZER_SHA,
        "initial_weights_sha": INITIAL_WEIGHTS_SHA,
        "config": CONFIG.to_dict(),
        "target_steps": TARGET_STEPS,
    }
    mismatches = [k for k, v in contract.items() if saved.get(k) != v]
    if mismatches:
        raise ValueError(
            "Existing distillation checkpoint does not match this run: "
            + ", ".join(mismatches)
            + ". Use a new RUN_NAME; old files are preserved."
        )
    model.load_state_dict(saved["model"], strict=True)
    optimizer.load_state_dict(saved["optimizer"])
    start_step = int(saved["step"])
    best_validation = float(saved.get("best_validation", float("inf")))
    best_step = int(saved.get("best_step", 0))
    stale_evaluations = int(saved.get("stale_evaluations", 0))
    best_model_state = saved.get("best_model_state")
    torch.set_rng_state(saved["torch_rng"])
    loader_gen.set_state(saved["loader_rng"])
    print("Resuming distillation checkpoint at step:", start_step)
    print("Best validation CE so far:", best_validation, "| best step:", best_step)

# Remove metrics written after the last verified checkpoint, if a Colab runtime stopped mid-run.
if metrics_path.is_file():
    retained = []
    for line in metrics_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        if int(item.get("step", 0)) <= start_step:
            retained.append(json.dumps(item))
    temp_metrics = metrics_path.with_suffix(".tmp")
    temp_metrics.write_text("\n".join(retained) + ("\n" if retained else ""), encoding="utf-8")
    os.replace(temp_metrics, metrics_path)

def evaluate():
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.inference_mode():
        for index, batch in enumerate(valid_loader):
            if index >= 12:
                break
            out = model(batch["input_ids"], thinking_steps=CONFIG.thinking_steps)
            labels = batch["labels"]
            count = int((labels != -100).sum())
            if count:
                loss_sum = F.cross_entropy(
                    out["logits"].float().reshape(-1, CONFIG.vocab_size),
                    labels.reshape(-1),
                    ignore_index=-100,
                    reduction="sum",
                )
                total_loss += float(loss_sum)
                total_tokens += count
    model.train()
    if total_tokens == 0:
        raise RuntimeError("Validation has no supervised assistant tokens.")
    return total_loss / total_tokens

def save_checkpoint(step):
    payload = {
        "format": "qmini-existing-drive-sft-v1",
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "config": CONFIG.to_dict(),
        "data_sha": DATASET_SHA,
        "tokenizer_sha": TOKENIZER_SHA,
        "initial_weights_sha": INITIAL_WEIGHTS_SHA,
        "target_steps": TARGET_STEPS,
        "best_validation": best_validation,
        "best_step": best_step,
        "stale_evaluations": stale_evaluations,
        "best_model_state": best_model_state,
        "torch_rng": torch.get_rng_state(),
        "loader_rng": loader_gen.get_state(),
    }
    temp = checkpoint_path.with_suffix(".tmp")
    torch.save(payload, temp)
    os.replace(temp, checkpoint_path)

iterator = iter(train_loader)
last_completed_step = start_step
for step in range(start_step, TARGET_STEPS):
    optimizer.zero_grad(set_to_none=True)
    micro_batches = []
    for _ in range(GRAD_ACCUM):
        try:
            micro_batches.append(next(iterator))
        except StopIteration:
            iterator = iter(train_loader)
            micro_batches.append(next(iterator))

    total_targets = sum(int((batch["labels"] != -100).sum()) for batch in micro_batches)
    if total_targets <= 0:
        raise RuntimeError("Batch contains no assistant target tokens.")

    ce_sum_total = 0.0
    aux_total = 0.0
    for batch in micro_batches:
        result = model(batch["input_ids"], thinking_steps=CONFIG.thinking_steps)
        ce_sum = F.cross_entropy(
            result["logits"].float().reshape(-1, CONFIG.vocab_size),
            batch["labels"].reshape(-1),
            ignore_index=-100,
            reduction="sum",
        )
        ce = ce_sum / total_targets
        aux = result["aux_loss"].float() / GRAD_ACCUM
        loss = ce + CONFIG.router_aux_loss_coef * aux
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at step {step + 1}")
        loss.backward()
        ce_sum_total += float(ce_sum.detach())
        aux_total += float(aux.detach())

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    warmup = max(1, int(TARGET_STEPS * 0.03))
    if step < warmup:
        learning_rate = LEARNING_RATE * (step + 1) / warmup
    else:
        progress = min(1.0, (step - warmup) / max(1, TARGET_STEPS - warmup))
        learning_rate = LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress)))
    for group in optimizer.param_groups:
        group["lr"] = learning_rate
    optimizer.step()

    completed = step + 1
    last_completed_step = completed
    train_ce = ce_sum_total / total_targets
    valid_ce = evaluate() if completed % 50 == 0 or completed == TARGET_STEPS else None
    if valid_ce is not None:
        if valid_ce < best_validation:
            best_validation = valid_ce
            best_step = completed
            stale_evaluations = 0
            best_model_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            print(f"New best validation CE: {best_validation:.4f} at step {best_step}")
        else:
            stale_evaluations += 1
    record = {
        "step": completed,
        "train_assistant_ce": train_ce,
        "validation_assistant_ce": valid_ce,
        "assistant_target_tokens": total_targets,
        "router_aux_loss": aux_total,
        "gradient_norm": float(grad_norm),
        "learning_rate": learning_rate,
        "device": "cpu",
        "synthetic_teacher_examples": 0,
    }
    with metrics_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

    if completed % 10 == 0 or completed == 1 or completed == TARGET_STEPS:
        print(
            f"step {completed}/{TARGET_STEPS} | "
            f"train CE={train_ce:.4f} | "
            f"validation CE={valid_ce if valid_ce is not None else 'pending'}"
        )
    if completed % 50 == 0 or completed == TARGET_STEPS:
        save_checkpoint(completed)
    if valid_ce is not None and stale_evaluations >= EARLY_STOP_PATIENCE:
        print(
            f"Early stopping: validation did not improve for {EARLY_STOP_PATIENCE} evaluations. "
            f"Best step was {best_step} with CE={best_validation:.4f}."
        )
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state, strict=True)
    print(f"Restored best validation checkpoint from step {best_step} (CE={best_validation:.4f}).")

model.eval()
final_state_path = RUN_DIR / "assistant_distill_model.pt"
torch.save(model.state_dict(), final_state_path)
print("Training finished. Model weights:", final_state_path)


In [ ]:
# Export and verify Q2-v2, then compare the trained QAT path with packed Q2 inference.
model.eval()
CONFIG.save(RUN_DIR / "config.json")
shutil.copy2(TOKENIZER_PATH, RUN_DIR / "tokenizer.json")

packed_path = RUN_DIR / "model.qmini.npz"
export_packed_state(model.state_dict(), CONFIG, packed_path)
packed_report = verify_packed_export(packed_path)
if packed_report.get("format") != "qmini-packed-v2":
    raise RuntimeError("Q2-v2 export verification failed.")
if packed_report.get("parameter_count") != CONFIG.estimated_parameter_count():
    raise RuntimeError("Packed export parameter count mismatch.")

qat_model = model
q2_model, q2_tokenizer, loaded_device = load_model(str(RUN_DIR), device="cpu")
q2_model.eval()

QUESTIONS = [
    "Hi",
    "How are you?",
    "What are you doing?",
    "Can you explain what a transformer is in one simple sentence?",
    "What is 2 plus 2?",
    "How can I learn English every day?",
]

def make_prompt(question):
    return (
        f"<|system|>{SYSTEM_PROMPT}<|end|>"
        f"<|user|>{question}<|end|>"
        f"<|assistant|>"
    )

def ask(candidate, candidate_tokenizer, question):
    return generate(
        candidate,
        candidate_tokenizer,
        make_prompt(question),
        max_new_tokens=64,
        temperature=0.0,
        top_k=1,
        top_p=1.0,
        thinking_steps=candidate.config.thinking_steps,
        serialized_prompt=True,
        frequency_penalty=0.25,
        presence_penalty=0.1,
    )

evaluation = []
for question in QUESTIONS:
    qat_answer = ask(qat_model, TOKENIZER, question)
    q2_answer = ask(q2_model, q2_tokenizer, question)
    evaluation.append({
        "question": question,
        "qat_fake_q2_answer": qat_answer,
        "packed_q2_answer": q2_answer,
    })
    print("\nUSER:", question)
    print("QAT model:", qat_answer)
    print("Packed Q2:", q2_answer)

report_path = RUN_DIR / "generation_evaluation.json"
report_path.write_text(json.dumps(evaluation, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

manifest = {
    "format": "qmini-existing-drive-sft-manifest-v1",
    "task": "human_conversation_sft_plus_local_drive_retrieval",
    "device": "cpu",
    "parameters": CONFIG.estimated_parameter_count(),
    "base_checkpoint": str(BASE_CHECKPOINT),
    "base_checkpoint_sha256": BASE_MODEL_SHA,
    "initial_weights_sha256": INITIAL_WEIGHTS_SHA,
    "tokenizer_sha256": TOKENIZER_SHA,
    "human_examples": len(human_records),
    "mistral_teacher_examples": 0,
    "teacher_model": None,
    "synthetic_examples_are_not_human": True,
    "external_api_used": False,
    "optimizer_steps_target": TARGET_STEPS,
    "optimizer_steps_completed": last_completed_step,
    "best_validation_ce": best_validation,
    "best_validation_step": best_step,
    "early_stop_patience_evaluations": EARLY_STOP_PATIENCE,
    "physical_batch": PHYSICAL_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch": EFFECTIVE_BATCH,
    "context_length": CONTEXT_LENGTH,
    "learning_rate_peak": LEARNING_RATE,
    "assistant_end_token_supervised": True,
    "packed_format": packed_report.get("format"),
    "packed_bytes": packed_path.stat().st_size,
    "packed_sha256": sha256_file(packed_path),
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print("\nQ-MINI DISTILLATION RUN COMPLETE")
print("Output folder:", RUN_DIR)
print("Packed Q2 model:", packed_path)
print("Evaluation report:", report_path)
print("Files:", sorted(p.name for p in RUN_DIR.iterdir() if p.is_file()))


## Drive books and datasets: local searchable library

इस चरण में supported files से searchable text index बनेगा। यह local CPU और SQLite से चलेगा; कोई बाहरी AI/API request नहीं होगी।

In [ ]:
# Build a local, disk-backed searchable library from existing Drive books and datasets.
import sqlite3, csv
from pathlib import Path

try:
    import pypdf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pypdf"], check=True)
    import pypdf
try:
    import docx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "python-docx"], check=True)
    import docx
try:
    import pyarrow.parquet as pq
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyarrow"], check=True)
    import pyarrow.parquet as pq

MYDRIVE = Path("/content/drive/MyDrive")
INDEX_DIR = PROJECT_DRIVE / "local_knowledge_library"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = INDEX_DIR / "qmini_knowledge.sqlite3"
SUPPORTED = {".txt", ".md", ".csv", ".json", ".jsonl", ".pdf", ".docx", ".parquet"}
SKIP_PARTS = {".trash", ".cache", ".ipynb_checkpoints", "node_modules", "training_state", "checkpoints", "optimizer", ".git"}


db = sqlite3.connect(DB_PATH)
db.execute("CREATE TABLE IF NOT EXISTS indexed_files(path TEXT PRIMARY KEY, mtime REAL, size INTEGER, chunks INTEGER)")
db.execute("CREATE VIRTUAL TABLE IF NOT EXISTS chunks USING fts5(content, source, tokenize='unicode61')")
db.commit()

def clean_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()

def flatten_json(obj):
    out = []
    if isinstance(obj, dict):
        preferred = ("text", "content", "body", "article", "passage", "paragraph", "title", "question", "answer", "summary")
        keys = [k for k in obj if any(p in str(k).lower() for p in preferred)]
        for k in (keys or list(obj.keys())):
            out.extend(flatten_json(obj[k]))
    elif isinstance(obj, list):
        for item in obj:
            out.extend(flatten_json(item))
    elif isinstance(obj, str) and len(obj.strip()) > 30:
        out.append(obj.strip())
    return out

def extract_file_text(path):
    ext = path.suffix.lower()
    if ext in {".txt", ".md"}:
        yield path.read_text(encoding="utf-8", errors="ignore")
    elif ext == ".csv":
        with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
            for row in csv.reader(f):
                yield " | ".join(x for x in row if x)
    elif ext in {".json", ".jsonl"}:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            if ext == ".json":
                try: yield from flatten_json(json.load(f))
                except Exception: return
            else:
                for line in f:
                    try: yield from flatten_json(json.loads(line))
                    except Exception: continue
    elif ext == ".pdf":
        reader = pypdf.PdfReader(str(path))
        for page in reader.pages: yield page.extract_text() or ""
    elif ext == ".docx":
        document = docx.Document(str(path))
        yield from (p.text for p in document.paragraphs if p.text.strip())
        for table in document.tables:
            for row in table.rows: yield " | ".join(cell.text for cell in row.cells)
    elif ext == ".parquet":
        parquet = pq.ParquetFile(path)
        cols = parquet.schema.names
        preferred = [c for c in cols if any(k in c.lower() for k in ("text", "content", "article", "body", "title", "summary", "document"))]
        for batch in parquet.iter_batches(batch_size=128, columns=preferred or cols):
            for row in batch.to_pylist():
                vals = [str(v) for v in row.values() if isinstance(v, str) and len(v.strip()) > 30]
                if vals: yield " ".join(vals)

def split_chunks(text, size=1200, overlap=180):
    text = clean_text(text)
    start = 0
    while start < len(text):
        part = text[start:start+size].strip()
        if len(part) >= 80: yield part
        if start + size >= len(text): break
        start += size - overlap

scan_roots = [MYDRIVE, PROJECT_DRIVE]  # Search all of MyDrive for supported book/data formats.
roots, seen_roots = [], set()
for root in scan_roots:
    key = str(root.resolve()) if root.exists() else str(root)
    if root.exists() and key not in seen_roots:
        roots.append(root); seen_roots.add(key)

files, seen_files = [], set()
for root in roots:
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in SUPPORTED: continue
        if any(part in SKIP_PARTS for part in p.parts): continue
        if p.stat().st_size > 2_000_000_000: continue
        key = str(p.resolve())
        if key not in seen_files: files.append(p); seen_files.add(key)

print("Supported source files discovered:", len(files))
for file_no, path in enumerate(files, 1):
    stat = path.stat()
    old = db.execute("SELECT mtime,size FROM indexed_files WHERE path=?", (str(path),)).fetchone()
    if old and old == (stat.st_mtime, stat.st_size): continue
    db.execute("DELETE FROM chunks WHERE source=?", (str(path),))
    count, batch_rows = 0, []
    try:
        for piece in extract_file_text(path):
            for chunk_text in split_chunks(piece):
                batch_rows.append((chunk_text, str(path)))
                if len(batch_rows) >= 500:
                    db.executemany("INSERT INTO chunks(content,source) VALUES(?,?)", batch_rows)
                    count += len(batch_rows); batch_rows.clear()
        if batch_rows:
            db.executemany("INSERT INTO chunks(content,source) VALUES(?,?)", batch_rows)
            count += len(batch_rows)
        db.execute("INSERT OR REPLACE INTO indexed_files(path,mtime,size,chunks) VALUES(?,?,?,?)",
                   (str(path), stat.st_mtime, stat.st_size, count))
        db.commit()
    except Exception as exc:
        db.rollback()
        print(f"Skipped {path.name}: {type(exc).__name__}: {str(exc)[:120]}")
    if file_no % 25 == 0: print(f"Scanned {file_no}/{len(files)} files")

file_count = db.execute("SELECT COUNT(*) FROM indexed_files").fetchone()[0]
chunk_count = db.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print("Indexed files:", file_count, "| searchable chunks:", chunk_count)
print("Local index:", DB_PATH)

def search_knowledge(question, limit=3):
    words = re.findall(r"[^\W_]{2,}", question, flags=re.UNICODE)[:12]
    if not words: return []
    query = " OR ".join('"' + w.replace('"', '""') + '"' for w in words)
    try:
        rows = db.execute("SELECT content,source FROM chunks WHERE chunks MATCH ? ORDER BY bm25(chunks) LIMIT ?",
                          (query, limit)).fetchall()
        return [{"content": r[0], "source": r[1]} for r in rows]
    except sqlite3.OperationalError:
        return []


In [ ]:
# Interactive local Q-Mini chat. Relevant Drive passages are retrieved locally.
q2_model.eval()
chat_history = []

def chat_prompt(question, passages):
    context_text = ""
    if passages:
        context_text = "\n\nReference passages from saved Drive files:\n"
        for i, item in enumerate(passages, 1):
            context_text += f"\n[Source {i}: {Path(item['source']).name}] {item['content'][:480]}"
        context_text += "\n\nUse these references when relevant. If they do not answer the question, say so. Do not invent book facts."
    history_text = ""
    for old_user, old_assistant in chat_history[-2:]:
        history_text += f"<|user|>{escape_special_token_literals(old_user)}<|end|><|assistant|>{escape_special_token_literals(old_assistant)}<|end|>"
    safe_question = escape_special_token_literals(question + context_text)
    return (f"<|system|>{SYSTEM_PROMPT} Use retrieved references accurately and mention source filenames when relevant.<|end|>"
            f"{history_text}<|user|>{safe_question}<|end|><|assistant|>")

print("\nQ-Mini is ready. Type 'exit' to stop or 'clear' to clear history.")
while True:
    user_text = input("\nYou: ").strip()
    if user_text.lower() in {"exit", "quit", "bye"}:
        print("Q-Mini chat closed."); break
    if user_text.lower() == "clear":
        chat_history = []; print("Chat history cleared."); continue
    if not user_text: continue
    references = search_knowledge(user_text, limit=3)
    answer = generate(q2_model, q2_tokenizer, chat_prompt(user_text, references),
                      max_new_tokens=96, temperature=0.25, top_k=20, top_p=0.88,
                      thinking_steps=q2_model.config.thinking_steps, serialized_prompt=True,
                      frequency_penalty=0.35, presence_penalty=0.15)
    print("\nQ-Mini:", answer.strip())
    if references:
        print("\nLocal references used:")
        for item in references: print("-", Path(item["source"]).name)
    chat_history.append((user_text, answer.strip()))
    chat_history = chat_history[-3:]


## नोट
- Mistral या किसी paid API का उपयोग नहीं हुआ।
- पुरानी किताबें searchable library में रहेंगी; model जरूरत पड़ने पर relevant अंश खोजेगा।
- 0.7M मॉडल की क्षमता सीमित है; बड़े मॉडल जैसी गुणवत्ता की गारंटी नहीं है।
- Training English-first है; अन्य भाषाओं की सामग्री अलग searchable reference रहेगी।